In [1]:
!pip install -q sentence-transformers huggingface_hub

In [2]:
from google.colab import files

print("📤 Veuillez sélectionner le fichier pairs_boosted.jsonl")
uploaded = files.upload()

# Vérifier que le fichier a bien été uploadé
import os
if "pairs_boosted.jsonl" not in uploaded:
    print("❌ Fichier pairs_boosted.jsonl non uploadé. Exécutez la cellule à nouveau.")
else:
    print(f"✅ Fichier pairs_boosted.jsonl uploadé avec succès ({len(uploaded['pairs_boosted.jsonl'])} octets)")

📤 Veuillez sélectionner le fichier pairs_boosted.jsonl


Saving pairs_boosted.jsonl to pairs_boosted.jsonl
✅ Fichier pairs_boosted.jsonl uploadé avec succès (226494 octets)


In [3]:
import json
from pathlib import Path
from sentence_transformers import SentenceTransformer, losses, InputExample
from torch.utils.data import DataLoader

# Charger les paires depuis le fichier uploadé
pairs_path = Path('pairs_boosted.jsonl')
examples = []
for line in pairs_path.read_text(encoding='utf-8').splitlines():
    if line.strip():
        row = json.loads(line)
        examples.append(InputExample(texts=[row['text_a'], row['text_b']]))

print(f"✅ {len(examples)} paires chargées")

# Modèle de base
BASE_MODEL = 'paraphrase-multilingual-MiniLM-L12-v2'
OUTPUT_DIR = 'training/output/everycli-minilm-ft-boosted'
EPOCHS = 4
BATCH_SIZE = 32

model = SentenceTransformer(BASE_MODEL)
train_dataloader = DataLoader(examples, shuffle=True, batch_size=BATCH_SIZE)
train_loss = losses.MultipleNegativesRankingLoss(model)

warmup_steps = int(len(train_dataloader) * EPOCHS * 0.1)

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    show_progress_bar=True,
)

model.save(OUTPUT_DIR)
print(f"✅ Modèle sauvegardé dans {OUTPUT_DIR}")

/tmp/ipykernel_549/1310012224.py:3: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, losses, InputExample


✅ 1880 paires chargées


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

...

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 0, 'pad_token_id': 1}.


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Modèle sauvegardé dans training/output/everycli-minilm-ft-boosted


In [4]:
from huggingface_hub import HfApi, login
from getpass import getpass

# Se connecter
token = getpass("Entrez votre token Hugging Face : ")
login(token=token)

# Uploader le modèle
REPO_ID = "Michelhe/everycli-minilm-ft-boosted"

api = HfApi()
try:
    api.create_repo(repo_id=REPO_ID, exist_ok=True)
    print(f"✅ Repo créé : {REPO_ID}")
except:
    print(f"ℹ️ Repo existe déjà : {REPO_ID}")

api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=REPO_ID,
    repo_type="model",
)

print(f"✅ Modèle poussé sur https://huggingface.co/{REPO_ID}")

Entrez votre token Hugging Face : ··········
✅ Repo créé : Michelhe/everycli-minilm-ft-boosted
✅ Modèle poussé sur https://huggingface.co/Michelhe/everycli-minilm-ft-boosted


In [5]:
from sentence_transformers import SentenceTransformer

model_test = SentenceTransformer(REPO_ID)
query = "Je veux mettre mon travail de côté sans faire de commit"
embedding = model_test.encode(query)
print(f"✅ Embedding généré (taille: {len(embedding)})")

modules.json:   0%|          | 0.00/277 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/284 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/12.1k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/241 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/742 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/588 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

✅ Embedding généré (taille: 384)
